# Chest X-Ray Report Generation (MIMIC-CXR)

DenseNet121 encoder (initialised from the trained multi-label classifier) + Transformer decoder.
The encoder already knows what the 13 findings look like, so the decoder starts from
diagnosis-aware visual features instead of plain ImageNet features.

Run order: helper files -> Setup -> Data -> Vocabulary -> Model -> Train -> Evaluate -> Samples

This version adds beam search decoding and drops label smoothing.

## 0. Create the helper modules

Run these two cells once. They write `rg_data.py` and `rg_metrics.py` next to the notebook.

In [ ]:
%%writefile rg_data.py
import json
import os
import re
from collections import Counter

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

IMG_SIZE = 224
MAX_LEN = 100
MIN_FREQ = 3

PAD, BOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]

_NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    _NORM,
])

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    _NORM,
])


def tokenize(text):
    t = str(text).lower()
    t = re.sub(r"[^a-z0-9%./\- ]", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip().split()


class Vocab:
    def __init__(self, itos):
        self.itos = list(itos)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens, max_len=MAX_LEN):
        ids = [BOS] + [self.stoi.get(t, UNK) for t in tokens][: max_len - 2] + [EOS]
        return ids + [PAD] * (max_len - len(ids))

    def decode(self, ids):
        words = []
        for i in ids:
            i = int(i)
            if i == EOS:
                break
            if i > UNK:
                words.append(self.itos[i])
        return words

    def save(self, path):
        json.dump(self.itos, open(path, "w"))

    @staticmethod
    def load(path):
        return Vocab(json.load(open(path)))

    @staticmethod
    def build(texts, min_freq=MIN_FREQ):
        c = Counter()
        for t in texts:
            c.update(tokenize(t))
        words = sorted([w for w, n in c.items() if n >= min_freq])
        return Vocab(SPECIALS + words), c


class ReportDataset(Dataset):
    def __init__(self, df, vocab, tf, max_len=MAX_LEN):
        self.paths = df["local_path"].tolist()
        self.reports = df["report"].tolist()
        self.vocab = vocab
        self.tf = tf
        self.max_len = max_len

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        for k in range(10):
            j = (i + k) % len(self.paths)
            try:
                img = Image.open(self.paths[j]).convert("RGB")
            except Exception:
                continue
            ids = self.vocab.encode(tokenize(self.reports[j]), self.max_len)
            return self.tf(img), torch.tensor(ids, dtype=torch.long)
        raise RuntimeError("too many unreadable images near index %d" % i)


def load_pairs(data_dir=".", one_image_per_study=True, val_fraction=0.05, seed=42):
    pairs = pd.read_csv(os.path.join(data_dir, "image_report_pairs.csv"))
    avail = pd.read_csv(os.path.join(data_dir, "available_labels.csv"))

    df = pairs.merge(avail[["dicom_id", "local_path"]], on="dicom_id", how="inner")
    df = df[df["report"].notna()]

    if one_image_per_study:
        df = df.sort_values("dicom_id").drop_duplicates(subset=["study_id"])

    df = df.reset_index(drop=True)

    train = df[df["split"] == "train"].reset_index(drop=True)
    val = df[df["split"] == "validate"].reset_index(drop=True)
    test = df[df["split"] == "test"].reset_index(drop=True)

    if len(val) < 500:
        extra = train.sample(n=int(len(train) * val_fraction), random_state=seed)
        train = train.drop(extra.index).reset_index(drop=True)
        val = pd.concat([val, extra], ignore_index=True)

    return train, val, test

In [ ]:
%%writefile rg_metrics.py
import math
from collections import Counter


def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def corpus_bleu(hyps, refs, max_n=4):
    num = [0] * max_n
    den = [0] * max_n
    hyp_len = 0
    ref_len = 0

    for h, r in zip(hyps, refs):
        hyp_len += len(h)
        ref_len += len(r)
        for n in range(1, max_n + 1):
            hn = _ngrams(h, n)
            rn = _ngrams(r, n)
            num[n - 1] += sum(min(c, rn[g]) for g, c in hn.items())
            den[n - 1] += max(sum(hn.values()), 0)

    scores = []
    log_sum = 0.0
    for n in range(max_n):
        p = (num[n] + 1e-9) / (den[n] + 1e-9)
        log_sum += math.log(p)
        bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
        scores.append(bp * math.exp(log_sum / (n + 1)))
    return scores


def _lcs(a, b):
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        cur = [0] * (n + 1)
        ai = a[i - 1]
        for j in range(1, n + 1):
            cur[j] = prev[j - 1] + 1 if ai == b[j - 1] else max(prev[j], cur[j - 1])
        prev = cur
    return prev[n]


def rouge_l(hyps, refs, beta=1.2):
    total = 0.0
    for h, r in zip(hyps, refs):
        if not h or not r:
            continue
        l = _lcs(h, r)
        if l == 0:
            continue
        p = l / len(h)
        rc = l / len(r)
        total += ((1 + beta ** 2) * p * rc) / (rc + beta ** 2 * p)
    return total / max(len(hyps), 1)


def evaluate_text(hyps, refs):
    b = corpus_bleu(hyps, refs)
    return {
        "BLEU-1": b[0], "BLEU-2": b[1], "BLEU-3": b[2], "BLEU-4": b[3],
        "ROUGE-L": rouge_l(hyps, refs),
    }

## 1. Setup

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.models import densenet121

from rg_data import (Vocab, ReportDataset, load_pairs, tokenize,
                     train_tf, eval_tf, PAD, BOS, EOS, MAX_LEN)
from rg_metrics import evaluate_text

DATA_DIR     = "."
CKPT_DIR     = "checkpoints_rg_final"
CLS_CKPT     = "checkpoints/best.pt"

BATCH_SIZE   = 16
EPOCHS       = 15
LR           = 1e-4
ENCODER_LR   = 1e-5
D_MODEL      = 512
N_HEAD       = 8
N_LAYERS     = 3
FF_DIM       = 2048
DROPOUT      = 0.1
NUM_WORKERS  = 4
EVAL_SUBSET  = 500
LABEL_SMOOTH = 0.0
BEAM         = 3
LEN_PENALTY  = 1.0
SEED         = 42

os.makedirs(CKPT_DIR, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

## 2. Image-report pairs

One image per study: a study's report describes the whole study, so keeping both the AP and
PA view would train the model on the same target text twice and inflate the scores.

In [ ]:
train_df, val_df, test_df = load_pairs(DATA_DIR)
print(f"train {len(train_df)}   val {len(val_df)}   test {len(test_df)}")

lens = train_df["report"].apply(lambda t: len(tokenize(t)))
print(f"\nReport length in tokens: mean {lens.mean():.1f}  median {lens.median():.0f}  "
      f"p90 {lens.quantile(0.9):.0f}  max {lens.max()}")
print(f"Truncated at MAX_LEN={MAX_LEN}: {(lens > MAX_LEN - 2).mean() * 100:.1f}% of reports")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(lens, bins=60, color="#3A6EA5")
ax.axvline(MAX_LEN, color="#C0504D", ls="--", label=f"MAX_LEN={MAX_LEN}")
ax.set_xlabel("Tokens per report"); ax.set_ylabel("Studies")
ax.set_title("Report length distribution"); ax.legend()
plt.tight_layout(); plt.show()

## 3. Vocabulary

In [ ]:
vocab_path = os.path.join(CKPT_DIR, "vocab.json")

if os.path.isfile(vocab_path):
    vocab = Vocab.load(vocab_path)
    print("Loaded existing vocabulary")
else:
    vocab, counts = Vocab.build(train_df["report"].tolist())
    vocab.save(vocab_path)
    print(f"Total distinct words in train: {len(counts)}")

print("Vocabulary size (min_freq=3):", len(vocab))

sample = tokenize(train_df['report'].iloc[0])[:12]
print("\nExample tokens :", sample)
print("Example ids    :", vocab.encode(sample)[:14])

In [ ]:
train_loader = DataLoader(ReportDataset(train_df, vocab, train_tf), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                          drop_last=True, persistent_workers=NUM_WORKERS > 0)
val_loader   = DataLoader(ReportDataset(val_df, vocab, eval_tf), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=NUM_WORKERS > 0)

xb, yb = next(iter(train_loader))
print("images:", tuple(xb.shape), " tokens:", tuple(yb.shape))

## 4. Model

The encoder outputs a 7x7 grid of 1024-d features, i.e. 49 visual tokens. Those are
projected to `D_MODEL` and used as the decoder's memory, so cross-attention can look at
different parts of the lung field while producing each word.

In [ ]:
class ReportGenerator(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, nhead=N_HEAD, layers=N_LAYERS,
                 ff=FF_DIM, dropout=DROPOUT, max_len=MAX_LEN):
        super().__init__()
        backbone = densenet121(weights=None)
        backbone.classifier = nn.Linear(backbone.classifier.in_features, 13)
        if os.path.isfile(CLS_CKPT):
            ck = torch.load(CLS_CKPT, map_location="cpu", weights_only=False)
            backbone.load_state_dict(ck["model"])
            print(f"Encoder initialised from {CLS_CKPT} (AUC {ck.get('auc', float('nan')):.4f})")
        else:
            print("Classifier checkpoint not found - encoder starts from random weights")

        self.features = backbone.features
        self.proj = nn.Linear(1024, d_model)
        self.img_pos = nn.Parameter(torch.zeros(1, 49, d_model))

        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.txt_pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        self.drop = nn.Dropout(dropout)

        layer = nn.TransformerDecoderLayer(d_model, nhead, ff, dropout,
                                           batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(layer, layers)
        self.out = nn.Linear(d_model, vocab_size)

        nn.init.normal_(self.img_pos, std=0.02)
        nn.init.normal_(self.txt_pos, std=0.02)

    def encode(self, img):
        f = self.features(img)
        f = torch.relu(f).flatten(2).transpose(1, 2)
        return self.proj(f) + self.img_pos

    def decode(self, memory, tgt_in):
        t = tgt_in.size(1)
        x = self.drop(self.emb(tgt_in) + self.txt_pos[:, :t])
        mask = torch.triu(torch.ones(t, t, dtype=torch.bool, device=tgt_in.device), 1)
        pad_mask = tgt_in == PAD
        h = self.decoder(x, memory, tgt_mask=mask, tgt_key_padding_mask=pad_mask)
        return self.out(h)

    def forward(self, img, tgt_in):
        return self.decode(self.encode(img), tgt_in)

    @torch.no_grad()
    def generate(self, img, max_len=MAX_LEN, beam=1, length_penalty=1.0):
        dev = img.device
        memory = self.encode(img)
        b = img.size(0)

        if beam <= 1:
            ids = torch.full((b, 1), BOS, dtype=torch.long, device=dev)
            done = torch.zeros(b, dtype=torch.bool, device=dev)
            for _ in range(max_len - 1):
                nxt = self.decode(memory, ids)[:, -1].argmax(-1)
                nxt[done] = PAD
                ids = torch.cat([ids, nxt.unsqueeze(1)], dim=1)
                done = done | (nxt == EOS)
                if done.all():
                    break
            return ids[:, 1:]

        n, d = memory.size(1), memory.size(2)
        V = self.out.out_features
        memory = memory.unsqueeze(1).expand(b, beam, n, d).reshape(b * beam, n, d)

        ids = torch.full((b * beam, 1), BOS, dtype=torch.long, device=dev)
        scores = torch.full((b, beam), -1e9, device=dev)
        scores[:, 0] = 0.0
        scores = scores.view(-1)
        fin = torch.zeros(b * beam, dtype=torch.bool, device=dev)
        ar = torch.arange(b, device=dev).unsqueeze(1)

        for _ in range(max_len - 1):
            lp = torch.log_softmax(self.decode(memory, ids)[:, -1].float(), -1)
            lp[fin] = -1e9
            lp[fin, PAD] = 0.0
            cand = (scores.unsqueeze(1) + lp).view(b, beam * V)
            top, idx = cand.topk(beam, dim=-1)
            bi = torch.div(idx, V, rounding_mode="floor")
            tok = idx % V
            g = (ar * beam + bi).view(-1)
            ids = torch.cat([ids[g], tok.view(-1, 1)], dim=1)
            scores = top.reshape(-1)
            fin = fin[g] | (tok.view(-1) == EOS)
            if fin.all():
                break

        lens = (ids != PAD).sum(1).clamp(min=1).float()
        best = (scores / lens.pow(length_penalty)).view(b, beam).argmax(-1)
        sel = torch.arange(b, device=dev) * beam + best
        return ids[sel][:, 1:]

In [ ]:
model = ReportGenerator(len(vocab)).to(device)

n_par = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_par / 1e6:.1f} M")

criterion = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=LABEL_SMOOTH)
optimizer = torch.optim.AdamW([
    {"params": model.features.parameters(), "lr": ENCODER_LR},
    {"params": [p for n, p in model.named_parameters() if not n.startswith("features.")], "lr": LR},
], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

## 5. Train

Teacher forcing: the decoder sees the true previous words and predicts the next one, so the
input is the report without its last token and the target is the report without `<bos>`.

In [ ]:
@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    for img, ids in loader:
        img, ids = img.to(device, non_blocking=True), ids.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(img, ids[:, :-1])
            loss = criterion(logits.reshape(-1, logits.size(-1)).float(), ids[:, 1:].reshape(-1))
        total += loss.item() * img.size(0); n += img.size(0)
    return total / max(n, 1)


@torch.no_grad()
def eval_text(model, df, limit=EVAL_SUBSET):
    model.eval()
    sub = df.iloc[:limit]
    loader = DataLoader(ReportDataset(sub, vocab, eval_tf), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    hyps, refs = [], []
    for img, ids in loader:
        img = img.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            gen = model.generate(img, beam=BEAM, length_penalty=LEN_PENALTY)
        for g, r in zip(gen.cpu(), ids):
            hyps.append(vocab.decode(g))
            refs.append(vocab.decode(r[1:]))
    return evaluate_text(hyps, refs), hyps, refs

In [ ]:
history  = []
best_b4  = 0.0
start_ep = 1
last_path = os.path.join(CKPT_DIR, "last.pt")

if os.path.isfile(last_path):
    ck = torch.load(last_path, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    start_ep, best_b4, history = ck["epoch"] + 1, ck["best_b4"], ck.get("history", [])
    print(f"Resumed from epoch {ck['epoch']}, best BLEU-4 {best_b4:.4f}")

In [ ]:
for epoch in range(start_ep, EPOCHS + 1):
    model.train()
    running, seen, t0 = 0.0, 0, time.time()

    for step, (img, ids) in enumerate(train_loader, 1):
        img, ids = img.to(device, non_blocking=True), ids.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(img, ids[:, :-1])
            loss = criterion(logits.reshape(-1, logits.size(-1)).float(), ids[:, 1:].reshape(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()

        running += loss.item() * img.size(0); seen += img.size(0)
        if step % 100 == 0:
            print(f"  ep{epoch} {step}/{len(train_loader)}  loss {running/seen:.4f}"
                  f"  {seen/(time.time()-t0):.1f} img/s")

    tr_loss = running / seen
    vl_loss = eval_loss(model, val_loader)
    scores, _, _ = eval_text(model, val_df)
    scheduler.step(scores["BLEU-4"])

    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": vl_loss,
                    "scores": scores, "minutes": (time.time() - t0) / 60})
    print(f"epoch {epoch}: train {tr_loss:.4f}  val {vl_loss:.4f}  "
          + "  ".join(f"{k} {v:.4f}" for k, v in scores.items())
          + f"  ({(time.time()-t0)/60:.1f} min)")

    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(), "best_b4": max(best_b4, scores["BLEU-4"]),
                "history": history}, last_path)

    if scores["BLEU-4"] > best_b4:
        best_b4 = scores["BLEU-4"]
        torch.save({"model": model.state_dict(), "scores": scores},
                   os.path.join(CKPT_DIR, "best.pt"))
        print(f"    new best saved (BLEU-4 {best_b4:.4f})")

json.dump(history, open(os.path.join(CKPT_DIR, "history.json"), "w"), indent=2)

## 6. Results

In [ ]:
history = json.load(open(os.path.join(CKPT_DIR, "history.json")))
ep = [h["epoch"] for h in history]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(ep, [h["train_loss"] for h in history], marker="o", label="train")
a1.plot(ep, [h["val_loss"] for h in history], marker="s", label="val")
a1.set_xlabel("Epoch"); a1.set_ylabel("Cross-entropy"); a1.set_title("Loss")
a1.legend(); a1.grid(alpha=.3)

for k in ["BLEU-1", "BLEU-4", "ROUGE-L"]:
    a2.plot(ep, [h["scores"][k] for h in history], marker="o", label=k)
a2.set_xlabel("Epoch"); a2.set_ylabel("Score"); a2.set_title("Validation text metrics")
a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(pd.DataFrame([{"epoch": h["epoch"], "train": round(h["train_loss"], 4),
                     "val": round(h["val_loss"], 4),
                     **{k: round(v, 4) for k, v in h["scores"].items()}}
                    for h in history]).to_string(index=False))

## 7. Test set and published baselines

In [ ]:
ck = torch.load(os.path.join(CKPT_DIR, "best.pt"), map_location=device, weights_only=False)
model.load_state_dict(ck["model"])

eval_df = test_df if len(test_df) >= 200 else val_df
label = "test" if len(test_df) >= 200 else "val (test split too small)"
scores, hyps, refs = eval_text(model, eval_df, limit=2000)

print(f"Evaluated on {label}, n={min(len(eval_df), 2000)}\n")
rows = [{"model": "ours", **{k: round(v, 3) for k, v in scores.items()}},
        {"model": "R2Gen (MIMIC)", "BLEU-1": 0.353, "BLEU-2": 0.218, "BLEU-3": 0.145,
         "BLEU-4": 0.103, "ROUGE-L": 0.277},
        {"model": "R2GenCMN (MIMIC)", "BLEU-1": 0.353, "BLEU-2": 0.218, "BLEU-3": 0.148,
         "BLEU-4": 0.106, "ROUGE-L": 0.278}]
print(pd.DataFrame(rows).to_string(index=False))

## 8. Sample outputs

Read a few of these yourself. The metrics cannot tell you whether the model has collapsed
into always writing the same normal-sounding report - only looking at the text can.

In [ ]:
import random
random.seed(0)
for i in random.sample(range(len(hyps)), 5):
    print("REF :", " ".join(refs[i]))
    print("GEN :", " ".join(hyps[i]))
    print("-" * 90)

uniq = len({" ".join(h) for h in hyps})
print(f"\nDistinct generated reports: {uniq} / {len(hyps)} ({uniq/len(hyps)*100:.1f}%)")

In [ ]:
from clinical_eval import clinical_efficacy, LABELS

hyp_texts = [" ".join(h) for h in hyps]
true = eval_df.iloc[:len(hyp_texts)][LABELS].fillna(0).replace(-1, 0).values

per_class, summary = clinical_efficacy(hyp_texts, true)
print(summary)
print(per_class.sort_values("support", ascending=False).to_string(index=False))